# Step 15.1 — Original Transformer Architecture：把 Seq2Seq、Attention 与 Block 放回 2017 Encoder–Decoder 全景图

到 Step 14，我们已经拆开学过 Transformer 的核心零件：

```text
Seq2Seq encoder / decoder roles
Attention
Q / K / V
causal self-attention
Multi-Head Attention
Residual + Norm
Feed-Forward MLP
one Transformer-style Block
```

现在第一次回到 **Original Transformer（2017）** 的完整 architecture。

最重要的历史事实是：

> **原始 Transformer 不是 decoder-only GPT。它仍然是一个 encoder–decoder Seq2Seq architecture，只是把 recurrent encoder / decoder 换成了 Attention-based stacks。**

本节只恢复这张全局图，并区分 encoder self-attention、decoder masked self-attention、encoder–decoder cross-attention。

本轮不进入 Step 15.2，不解释为什么 GPT 可以删除 encoder。


## 1. 从 RNN Seq2Seq 到 Transformer：任务结构没有先消失

Step 10.1 的 recurrent Seq2Seq 是：

```text
source sequence
      ↓
   Encoder
      ↓
source representation
      ↓
   Decoder
      ↓
target sequence
```

2017 Transformer 仍然保留：

```text
Source side
   ↓
Encoder stack
   ↓
Encoder memory
   ↓
Decoder stack
   ↓
Target-side output
```

真正改变的是内部 computation：

```text
RNN recurrence
      ↓ replaced by
Attention + position-wise FFN + residual/norm
```

所以历史链条不是：

```text
Seq2Seq -> GPT
```

而是：

```text
RNN Seq2Seq
-> RNN Seq2Seq + Attention
-> Transformer Encoder–Decoder
-> later decoder-only GPT family
```


## 2. Original Transformer 的 Top-Level Data Flow

用课程当前记号：

```text
source token IDs                       target input IDs (shifted right)
    (B,T_src)                                  (B,T_tgt)
       │                                           │
       ↓                                           ↓
source embedding + position info       target embedding + position info
       │                                           │
       ↓                                           ↓
┌─────────────────┐                    ┌──────────────────────────┐
│  Encoder Layer   │                    │ masked self-attention    │
│  × N             │                    │          ↓               │
│                  │                    │ cross-attention          │
│ self-attention   │                    │          ↓               │
│       ↓          │                    │ position-wise FFN        │
│      FFN         │                    │          × N             │
└────────┬────────┘                    └─────────────┬────────────┘
         │                                          │
         ↓                                          ↓
 encoder memory                               decoder states
 (B,T_src,C)                                  (B,T_tgt,C)
         │                                          │
         └──────────── K / V ───────────────────────┤
                                                    ↓
                                          output projection / softmax
                                                    ↓
                                             target predictions
```

Position information 在原始 Transformer 中是必要组成，因为 recurrence 已被移除；本节只在架构图中标出，不展开具体 position mechanism。


## 3. Encoder Stack：Source Positions 彼此做非因果 Self-Attention

Encoder 输入：

$$
X_{src}\in\mathbb R^{B\times T_{src}\times C}.
$$

每个 encoder layer 的核心是：

```text
Multi-Head Self-Attention
↓
Residual / Norm
↓
Position-Wise FFN
↓
Residual / Norm
```

这里的 Self-Attention：

```text
Q <- source representations
K <- source representations
V <- source representations
```

而且 source sequence 在 encoder 时已经完整可见，所以它**不是 decoder-style causal mask**。

忽略 padding mask 等工程细节时，一个 source position 可以与所有 source positions 交互。

经过 N 层 encoder stack 后得到：

$$
\boxed{
E\in\mathbb R^{B\times T_{src}\times C}
}.
$$

这里把 E 称为 **encoder memory / encoder output representations**。


## 4. Decoder Stack：每层比 Encoder 多一个 Cross-Attention Sublayer

Decoder 输入来自 shifted target-side sequence：

$$
X_{tgt}\in\mathbb R^{B\times T_{tgt}\times C}.
$$

每个 original Transformer decoder layer 有三类核心 sublayers：

```text
1. Masked Multi-Head Self-Attention
2. Encoder–Decoder Cross-Attention
3. Position-Wise Feed-Forward Network
```

### 1. Masked self-attention

target-side positions 只能访问：

$$
j\le i.
$$

这就是我们 Step 12 已经学过的 causal rule。

### 2. Cross-attention

decoder 当前 representations 去读取 encoder memory。

### 3. FFN

对每个 target position 独立做 nonlinear feature transformation。

因此 original decoder layer 并不是我们 Step 14.5 那个只有 causal self-attention + MLP 的 block：

> **原始 encoder–decoder Transformer 的 decoder 还多一个 cross-attention sublayer。**


## 5. 三种 Attention 放在一起比较

| Attention 类型 | Query 来源 | Key / Value 来源 | Causal? | Score shape |
|---|---|---|---|---|
| Encoder self-attention | source | source | 通常否 | `(B,H,T_src,T_src)` |
| Decoder masked self-attention | target decoder states | target decoder states | 是 | `(B,H,T_tgt,T_tgt)` |
| Encoder–decoder cross-attention | decoder side | encoder memory | 对 source 无 decoder-style causal mask | `(B,H,T_tgt,T_src)` |

最重要的新 shape 是 cross-attention：

$$
\boxed{
(T_{tgt}\times T_{src})
}
$$

它不要求是方阵。

第 `i` 个 target Query 会对所有 source Keys 计算 compatibility。

这正是 Step 10 Attention intuition 的 Transformer / QKV 版本。


## 6. Cross-Attention 的 Q / K / V 来源

设当前 decoder representations：

$$
D\in\mathbb R^{B\times T_{tgt}\times C}
$$

encoder memory：

$$
E\in\mathbb R^{B\times T_{src}\times C}.
$$

Cross-attention 中：

$$
Q=DW_Q
\in
\mathbb R^{B\times T_{tgt}\times d_k},
$$

但：

$$
K=EW_K
\in
\mathbb R^{B\times T_{src}\times d_k},
$$

$$
V=EW_V
\in
\mathbb R^{B\times T_{src}\times d_v}.
$$

所以：

$$
QK^T
:
(B,T_{tgt},d_k)
@
(B,d_k,T_{src})
\rightarrow
\boxed{(B,T_{tgt},T_{src})}.
$$

再：

$$
A V
:
(B,T_{tgt},T_{src})
@
(B,T_{src},d_v)
\rightarrow
(B,T_{tgt},d_v).
$$

这就是：

> target-side Query 从 source-side memory 中读取 information payload。


In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn

B = 1
T_src = 3
T_tgt = 2
C = 4

# Architecture-level example only.
encoder_memory = mx.zeros((B, T_src, C))
decoder_state = mx.zeros((B, T_tgt, C))

q_proj = nn.Linear(input_dims=C, output_dims=C, bias=False)
k_proj = nn.Linear(input_dims=C, output_dims=C, bias=False)
v_proj = nn.Linear(input_dims=C, output_dims=C, bias=False)

q = q_proj(decoder_state)       # (B,T_tgt,C)
k = k_proj(encoder_memory)      # (B,T_src,C)
v = v_proj(encoder_memory)      # (B,T_src,C)

scores = mx.matmul(
    q,
    mx.swapaxes(k, -1, -2),
) / math.sqrt(C)                # (B,T_tgt,T_src)

weights = mx.softmax(scores, axis=-1)
context = mx.matmul(weights, v) # (B,T_tgt,C)

print('q:', q.shape)
print('k:', k.shape)
print('v:', v.shape)
print('cross-attention scores:', scores.shape)
print('cross-attention context:', context.shape)

# Expected shapes:
# q       = (1,2,4)
# k / v   = (1,3,4)
# scores  = (1,2,3)
# context = (1,2,4)
#
# No runtime result is claimed; the master notebook remains unexecuted.


## 7. Decoder Training 仍然需要 Shifted Target + Causal Mask

Step 10.1 已经见过：

```text
full target:
<BOS> A B C <EOS>

decoder input:
<BOS> A B C

prediction target:
A     B C <EOS>
```

Original Transformer 虽然移除了 decoder recurrence，但训练时完整 target-input tensor 会同时存在。

因此必须再用 masked self-attention 保证：

$$
\boxed{j\le i}.
$$

也就是：

> parallel training computation 不等于允许 target position i 读取 future target token。

Cross-attention 则不同：source sequence 已经完整提供给 encoder，所以当前 target Query 可以读取整条 encoder memory（除 padding 等 mask）。


## 8. 很重要：我们的 Step 14.5 Block 并不等于 2017 原始 Decoder Layer

为了教学与后续 MiniGPT，我们 Step 14.5 实现的是一个更现代的 decoder-style block：

```text
Pre-Norm
RMSNorm
causal self-attention
GELU MLP
no cross-attention
```

而 2017 原始 Transformer 的经典 architecture 使用：

```text
Encoder + Decoder
Post-Norm / Add & Norm
LayerNorm
ReLU position-wise FFN
decoder has masked self-attention + cross-attention
```

所以需要明确区分：

$$
\boxed{
\text{Transformer concept}
\neq
\text{one frozen implementation recipe}
}.
$$

核心思想在后续模型中被大量保留，但 normalization、activation、stack composition 等实现细节会演化。


## 9. Tiny Translation-Style Shape Walkthrough

假设：

```text
B = 2
T_src = 5
T_tgt = 4
C = 8
```

Source side：

$$
(2,5)
\rightarrow
\text{source embedding + position info}
\rightarrow
(2,5,8)
\rightarrow
\text{encoder stack}
\rightarrow
E:(2,5,8).
$$

Target side shifted input：

$$
(2,4)
\rightarrow
\text{target embedding + position info}
\rightarrow
(2,4,8).
$$

Decoder masked self-attention：

$$
(2,4,8)
\rightarrow
(2,4,8).
$$

Cross-attention compatibility matrix：

$$
\boxed{(2,H,4,5)}.
$$

Cross-attention output：

$$
(2,4,8).
$$

经过 decoder stack 后：

$$
(2,4,8)
$$

再映射到 target vocabulary logits。

注意 `T_src=5` 与 `T_tgt=4` 完全不需要相等。


In [ ]:
# SHAPE WALKTHROUGH ONLY — no encoder/decoder stack is executed.

B = 2
T_src = 5
T_tgt = 4
C = 8
H = 2

source_repr_shape = (B, T_src, C)
encoder_memory_shape = (B, T_src, C)
target_repr_shape = (B, T_tgt, C)
decoder_self_attn_shape = (B, T_tgt, C)
cross_score_shape = (B, H, T_tgt, T_src)
decoder_output_shape = (B, T_tgt, C)

print('source representations:', source_repr_shape)
print('encoder memory:', encoder_memory_shape)
print('target representations:', target_repr_shape)
print('decoder masked-self output:', decoder_self_attn_shape)
print('cross-attention scores:', cross_score_shape)
print('decoder output:', decoder_output_shape)

# These are architecture-level expected shapes, not runtime validation.


## 10. Transformer 相比 Recurrent Seq2Seq 真正改变了什么？

回到历史链：

### Recurrent Seq2Seq

Encoder / decoder 内部都有：

$$
h_t=f(x_t,h_{t-1})
$$

这样的 time-axis recurrence。

### Original Transformer

仍保留 encoder / decoder 的任务角色，但用：

- self-attention；
- cross-attention；
- position-wise FFN；
- residual / normalization；
- explicit positional information

替代 recurrent state transition。

这使同一层中多个 positions 的训练计算可以更充分并行，并显著缩短远距离 positions 间的 dependency path。

但 autoregressive decoder 的 generation 仍然是逐 token 进行；architecture 去掉 recurrence 不等于生成过程可以无条件一次性产生未来 tokens。


## 11. 为什么到这里仍然不能叫 GPT？

Original Transformer 仍然明确拥有：

```text
separate source sequence
↓
encoder stack
↓
encoder memory

separate target stream
↓
decoder stack
↓
cross-attention reads encoder memory
```

也就是说 decoder 的职责仍然是：

> 在 source encoder context 的条件下生成 target。

而 GPT 的关键结构变化之一，正是如何从这套 encoder–decoder architecture 走到：

```text
one token stream
+ causal self-attention
+ decoder-only stack
```

但“为什么这样就足以做 autoregressive language modeling”属于 Step 15.2。

本节到此停止。


## 12. Step 15.1 材料边界

- 2017 Original Transformer 是 encoder–decoder Seq2Seq architecture，不是 decoder-only GPT。
- Encoder stack 使用 source-side self-attention + position-wise FFN；source self-attention 通常不是 decoder-style causal attention。
- Decoder layer 包含 masked target self-attention、encoder–decoder cross-attention、position-wise FFN。
- Cross-attention 的 Q 来自 decoder，K/V 来自 encoder memory；score shape 为 `(B,H,T_tgt,T_src)`，不要求是方阵。
- Decoder training 仍使用 shifted target input 与 causal masking，防止 future target leakage；cross-attention 可以读取完整 source memory。
- Original Transformer 经典设计是 Post-Norm / LayerNorm / ReLU FFN；本课程 14.5 的 Pre-Norm / RMSNorm / GELU causal Block 是后续现代化教学选择，不能混为原版。
- 移除 recurrence 改善 training-time parallelism 与 dependency path，但 autoregressive generation 仍是逐 token 过程。
- 本轮只准备 Step 15.1；不进入 Step 15.2，不解释为什么 autoregressive GPT 可以删除 separate encoder，也不实现 decoder-only GPT。
